## Intermediate Data Science

## Important Information

- Email: [joanna_bieri@redlands.edu](mailto:joanna_bieri@redlands.edu)
- Office Hours take place in Duke 209 -- [Office Hours Schedule](https://joannabieri.com/schedule.html)
- [Class Website](https://joannabieri.com/data201_intermediate.html)
- [Syllabus](https://joannabieri.com/data201/IntermediateDataScience.pdf)

## Today's Reading

*Introduction to Machine Learning with Python*, Chapter 2.3.4 - Naive Bayes Classifiers.

## Final Project

Your **Project Proposal and Exploratory Data Analysis** is due **Tuesday 11/10 at 11:59pm**. It is individual work, on Canvas and in your sandbox. See the [Final Project Information](https://joannabieri.com/data201/FinalProjectInfo.html) page.

In [ ]:
# NOTE - This list of package imports is getting long
# In a professional setting you would only want to 
#      import what you need!
# I had chatGPT break the packages into groups here

# ============================================================
# Basic packages
# ============================================================
import os                             # For file and directory operations
import numpy as np                    # For numerical computing and arrays
import pandas as pd                   # For data manipulation and analysis

# ============================================================
# Visualization packages
# ============================================================
import matplotlib.pyplot as plt        # Static 2D plotting
import seaborn as sns                  # Statistical data visualization built on matplotlib

# Interactive visualization with Plotly
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = 'notebook_connected'   # Set renderer for interactive output in notebooks

# ============================================================
# Scikit-learn: Core utilities for model building and evaluation
# ============================================================
from sklearn.model_selection import train_test_split    # Train/test data splitting
from sklearn.preprocessing import PolynomialFeatures, MinMaxScaler, StandardScaler  # Feature transformations and scaling
from sklearn.metrics import (                            # Model evaluation metrics
    mean_squared_error, r2_score, accuracy_score, 
    precision_score, recall_score, confusion_matrix, 
    classification_report
)

# ============================================================
# Scikit-learn: Linear and polynomial models
# ============================================================
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge, Lasso
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor       # For KNN

# ============================================================
# Scikit-learn: Synthetic dataset generators
# ============================================================
from sklearn.datasets import make_classification, make_regression

# ============================================================
# Scikit-learn: Naive Bayes models
# ============================================================
from sklearn.naive_bayes import GaussianNB, BernoulliNB, MultinomialNB

In [ ]:
def plot_decision_boundary(model, X, y, title="Decision Boundary"):
    # A grid of points that covers the data
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                         np.linspace(y_min, y_max, 200))

    # Ask the model for a prediction at every point on the grid and shade the two classes
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    plt.contourf(xx, yy, Z, levels=[-1, 0.5, 1],
                 colors=['steelblue', 'darkorange'], alpha=0.2)

    # The data on top
    plt.scatter(X[y == 0, 0], X[y == 0, 1], color='steelblue', label='Class 0', alpha=0.7)
    plt.scatter(X[y == 1, 0], X[y == 1, 1], color='darkorange', label='Class 1', alpha=0.7)

    plt.title(title)
    plt.xlabel("Feature 1")
    plt.ylabel("Feature 2")
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.show()

## Naive Bayes Classifiers

Today we are going to talk about a family of models:

- Naive Bayes classifiers (Gaussian, Bernoulli, and Multinomial)

These models are used for a variety of tasks in machine learning. Naive Bayes classifiers are fairly simple and have strong assumptions but they can perform surprisingly well for high dimensional data. They are very fast, but handle nonlinear boundaries poorly.

**Reasons to Use Naive Bayes Classifiers:**

1. They work well with text data

Naive Bayes is one of the best and simplest methods for tasks like spam detection, sentiment analysis, and document classification.
It handles large vocabularies easily and performs well even when there are thousands of features (e.g., words).

2. They are fast and efficient

Training and prediction are both very quick because the algorithm only needs to count how often each feature occurs in each class.
This makes it great for real-time applications or very large datasets.

3. They perform surprisingly well even with simple assumptions

Even though Naive Bayes assumes that all features are independent once you know the class (which is rarely true in real data), it often gives accurate results even when this assumption is broken, especially when the features don't depend too much on each other.

4. They need little data to get started

Because the model is based on simple probability estimates, it can perform reasonably well even when you have small training datasets.

5. They are easy to interpret and implement

You can understand how the model makes decisions by looking at which features contribute most to each class.
This makes it a good teaching and baseline model.

6. They handle noisy or missing data fairly well

Since predictions are based on overall probability patterns, missing or slightly incorrect feature values usually don't drastically change the result.

7. They can serve as strong baselines

In many machine learning projects, Naive Bayes is used as a baseline model. They serve as a simple first approach: easy to compare more complex models against (like decision trees or neural networks).


## Understanding Naive Bayes

Naive Bayes classifiers use **probabilities** to decide which class something most likely belongs to.  They start with what we already know about how common each class is, and then **update those beliefs** based on the features of new data we see.

## How It Works

1. **Start with prior knowledge** 

Before looking at the data, the model already has an idea of how frequent each class is.  
   
*Example:* Maybe 80% of emails are "not spam" and 20% are "spam."

2. **Look at the evidence**  

The model checks which features appear (like certain words in an email).  It knows from past examples how common each feature is for each class.

3. **Combine the clues**  

It combines all the clues to see which class is *most likely* for this new data.  It assumes that all clues are independent once you know the class - that's why it's called **Naive** Bayes.


4. **Pick the most likely class**  
   Finally, it chooses the class with the highest overall probability.



## Example: Spam Filter

Imagine we're trying to classify an email as **spam** or **not spam**.

- The word **free** appears often in spam.  
- The word **meeting** appears often in normal emails.  

If a new email says **Free meeting this Friday!**, Naive Bayes looks at both clues and estimates which class (spam or not) is more likely, based on what it has learned.

Even if the email is a mix of both words, the model uses probability to make the best guess.

## Example: The Guessing Game

Think of Naive Bayes as a detective guessing what kind of event a person is attending, just from what they are wearing.

| Clue | Beach | Wedding | Business Meeting |
|------|--------|----------|------------------|
| Wearing sunglasses | ✅ | ❌ | ❌ |
| Wearing formal shoes | ❌ | ✅ | ✅ |
| Wearing a tie | ❌ | ✅ | ✅ |

If someone is wearing **sunglasses** and **formal shoes** then the detective uses both clues to decide - maybe it's most likely a **wedding**, even though either clue alone could fit more than one event.  

That's exactly what Naive Bayes does:  
It combines independent clues to make the most probable guess about the class.


## Naive Bayes Classifiers

Naive Bayes classifiers are probabilistic models based on **Bayes' theorem**, which relates the conditional and marginal probabilities of random events.

Notation: 

$P(x)$ is the probability of observation/input $x$ and $P(C_k)$ is the probability of class $C_k$, these are just normal probabilities. What is the chance of randomly drawing input $x$ or class $C_k$ from a bag?

The conditional probability, $ P(x | C_k) $, is the probability of the observation being $x$ given that we are looking in class $C_k$ and we say "probability of $x$ given $C_k$". 

We can flip the conditional probability using Bayes' Theorem

$$
P(C_k | x) = \frac{P(x | C_k) P(C_k)}{P(x)}
$$

where:
- $ P(C_k | x) $: posterior probability of class $ C_k $ given input $ x $
- $ P(x | C_k) $: likelihood of the data given the class
- $ P(C_k) $: prior probability of the class
- $ P(x) $: evidence term (same for all classes)

We can solve for any term in the theorem if we know the other parts.

The "naive" assumption is that the features are **conditionally independent** given the class. So given a bunch of observations $x_i$ for class $C_k$ we can get the total conditional probability by just multiplying them.

$$
P(x | C_k) = \prod_{i=1}^n P(x_i | C_k)
$$

There are different assumptions that we can make about the underlying probability distributions - aka how the data is spread out.


## Gaussian Naive Bayes

Used when features are continuous and approximately normally distributed. You should try to check for a normal distribution in your data, but even data that breaks this rule slightly can still work.

**Strengths**
- Works well for continuous features with approximately normal distributions.
- Very fast and requires little training data.

**Weaknesses**
- Assumes independence among features.
- Sensitive to features that are not normally distributed.

**Key parameter**
- `var_smoothing`: small positive number added to variances for numerical stability.

In [ ]:
# Generate synthetic 2D data
X, y = make_classification(n_samples=400, n_features=2, n_classes=2,
                           n_redundant=0, n_clusters_per_class=1, random_state=42)

# Create the plot
plt.figure(figsize=(8, 6))
plt.scatter(X[y == 0, 0], X[y == 0, 1], color='steelblue', label='Class 0', alpha=0.7)
plt.scatter(X[y == 1, 0], X[y == 1, 1], color='darkorange', label='Class 1', alpha=0.7)

# Add labels and legend
plt.title("Synthetic Binary Classification Data")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

In [ ]:
df = pd.DataFrame(X,columns=['Feature 1', 'Feature 2'])
df['Label'] = y
df

In [ ]:
## Look for Gaussian distribution
continuous_numeric_features = ['Feature 1','Feature 2']
for feature in continuous_numeric_features:
    plt.figure(figsize=(4, 4))  # smaller figure size
    sns.histplot(df[feature],kde=True)
    plt.show()

In [ ]:
#Test train split/
X_train, X_test, y_train, y_test = train_test_split(X, y, 
                                                    test_size=0.2, 
                                                    random_state=42)

In [ ]:
gnb = GaussianNB()
gnb.fit(X_train, y_train)
y_pred = gnb.predict(X_test)

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

In [ ]:
plot_decision_boundary(gnb, X, y, "Gaussian Naive Bayes Decision Boundary")

In [ ]:
y_pred = gnb.predict(X)
print("Accuracy:", accuracy_score(y, y_pred))

## Bernoulli Naive Bayes

The Bernoulli Naive Bayes model works well with binary features, though it may struggle when features are correlated. The model assumes that each feature can take only two values (e.g., 0 or 1) and learns how likely each feature is to appear in each class.

It is most useful when **the presence or absence of a feature** matters more than how many times it occurs.

1. Text classification with binary word features
   - When you care whether a word appears in a document, not how many times.  
   - Example: Email spam detection using word present = 1 / word absent = 0.

2. Document or topic classification 
   - News or article classification using binary bag-of-words representation.

3. Feature sets with yes/no indicators 
   - Customer behavior: whether a person clicked an ad (1) or not (0).  
   - Medical diagnosis: whether a symptom is present (1) or absent (0).


**When *Not* to Use It**

- If your features are **counts** (e.g., word frequencies), use **Multinomial Naive Bayes** instead.  
- If your features are **continuous numeric values** (e.g., measurements, sensor data), use **Gaussian Naive Bayes**.

In [ ]:
np.random.seed(42)
n_samples = 200
X = np.random.randint(0, 2, (n_samples, 2))

# Define labels
# Class 1 if either feature is 1, else Class 0
y = np.logical_or(X[:, 0], X[:, 1]).astype(int)

In [ ]:
df = pd.DataFrame(X,columns=['Feature 1', 'Feature 2'])
df['Label - OR'] = y
df

In [ ]:
plt.figure(figsize=(6, 5))
plt.scatter(X[y == 0, 0], X[y == 0, 1], color='steelblue', label='Class 0', s=80)
plt.scatter(X[y == 1, 0], X[y == 1, 1], color='darkorange', label='Class 1', s=80)

plt.title("Simple OR Relationship")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.xticks([0, 1])
plt.yticks([0, 1])
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.show()

In [ ]:
bnb = BernoulliNB()
bnb.fit(X, y)
plot_decision_boundary(bnb, X, y, "Bernoulli Naive Bayes Decision Boundary")

In [ ]:
y_pred = bnb.predict(X)
print("Accuracy:", accuracy_score(y, y_pred))

Naive Bayes gets every point right here! But be careful about why. Look at the points in Class 1: if Feature 1 is 0 then Feature 2 HAS to be 1. So inside Class 1 the features are not independent, and the naive assumption is actually broken.

It still works because the OR points can be split by a straight line, and Bernoulli Naive Bayes always draws a straight line boundary. The independence assumption doesn't have to be exactly true, the model just needs the right class to get the biggest score.

## Let's break the algorithm!

In [ ]:
np.random.seed(42)
n_samples = 200
X = np.random.randint(0, 2, (100, 2))
y = np.logical_xor(X[:, 0], X[:, 1]).astype(int)  # XOR pattern

In [ ]:
df = pd.DataFrame(X,columns=['Feature 1', 'Feature 2'])
df['Label - XOR'] = y
df

In [ ]:
# Plot the data
plt.figure(figsize=(6, 5))
plt.scatter(X[y == 0, 0], X[y == 0, 1], color='steelblue', label='Class 0', s=80)
plt.scatter(X[y == 1, 0], X[y == 1, 1], color='darkorange', label='Class 1', s=80)

plt.title("XOR Pattern with Binary Features")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.xticks([0, 1])
plt.yticks([0, 1])
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.show()

In [ ]:
bnb = BernoulliNB()
bnb.fit(X, y)
plot_decision_boundary(bnb, X, y, "Bernoulli Naive Bayes Decision Boundary")

In [ ]:
y_pred = bnb.predict(X)
print("Accuracy:", accuracy_score(y, y_pred))

Naive Bayes fails here. In XOR the class depends on the two features together. Feature 1 by itself tells you nothing about the class, and Feature 2 by itself tells you nothing, you need both. Naive Bayes looks at one feature at a time and then multiplies, so it can't see this pattern. There is also no straight line that splits the XOR points, so a straight line boundary will get a lot of points wrong.

## Correlation Heatmap

Remember the naive assumption is that the features are independent **within each class** (conditional independence). This is different from the features being independent of each other overall! A regular correlation heatmap can't check this assumption, but it does tell us useful things about our features. Here are our two examples:

In [ ]:
np.random.seed(42)
n_samples = 200
X = np.random.randint(0, 2, (n_samples, 2))
y = np.logical_or(X[:, 0], X[:, 1]).astype(int)

# Put the data into a data frame:
df = pd.DataFrame(X, columns=['Feature 1', 'Feature 2'])
df['Target'] = y

# Compute correlation matrix
corr = df.corr()

plt.figure(figsize=(4, 3))
sns.heatmap(corr, annot=True, cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Correlation Heatmap for OR Data')
plt.show()

In the first example using **OR** the two features have low correlation with each other, and each one is fairly correlated with the target (about 0.6). So each feature is useful on its own.

In [ ]:
np.random.seed(42)
n_samples = 200
X = np.random.randint(0, 2, (100, 2))
y = np.logical_xor(X[:, 0], X[:, 1]).astype(int)  # XOR pattern

# Put the data into a data frame:
df = pd.DataFrame(X, columns=['Feature 1', 'Feature 2'])
df['Target'] = y

# Compute correlation matrix
corr = df.corr()

plt.figure(figsize=(4, 3))
sns.heatmap(corr, annot=True, cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Correlation Heatmap for XOR Data')
plt.show()

In the second example using **XOR** the two features also have low correlation with each other, but neither one is correlated with the target at all. Each feature is useless on its own. Together they decide the output.

So the two heatmaps look about the same for the features, but Naive Bayes only works on one of them. To check the naive assumption we need to look INSIDE each class.

In [ ]:
# Check the naive assumption: are the features independent INSIDE each class?
# df is still the XOR data from above
for label in [0, 1]:
    # keep only the rows in this class
    df_class = df[df['Target'] == label]
    # correlation between the two features, inside this class only
    r = df_class['Feature 1'].corr(df_class['Feature 2'])
    print('XOR Class', label, '- correlation between the features:', round(r, 2))

Inside each class the features are perfectly correlated! In Class 0 the two features are always the same, (0,0) or (1,1), and in Class 1 they are always different, (0,1) or (1,0). This is as far from independent as you can get, and it is why Naive Bayes fails.

NOTE - If you try this on the OR data, Class 0 only has the point (0,0), so there is nothing to correlate and pandas gives you NaN. Class 1 has a correlation of about -0.5, so the assumption is broken there too, even though the model gets every point right. In real data the assumption is almost never exactly true. What you want to watch out for is features that are VERY strongly related inside a class.

## Multinomial Naive Bayes

Multinomial Naive Bayes is best used for data where the features represent **discrete counts or frequencies**, such as how many times a word appears in a document or how many events of a certain type occur.

It assumes that:
- Each feature represents a **count** (non-negative integer).  
- Features are **conditionally independent** given the class.  
- The total number of counts can vary across samples.


1. **Text classification using word counts**
   - Example: Spam detection, news categorization, or sentiment analysis.
   - Each feature represents how often a word appears in a document.

2. **Document or topic modeling**
   - Assign documents to topics (e.g., sports, politics, science).
   - Each topic has a characteristic distribution of word frequencies.

3. **Bag-of-words models in Natural Language Processing (NLP)**
   - Suitable when features are integer counts (not just presence/absence).

4. **Other count-based features**
   - Customer behavior: number of purchases, clicks, or actions of different types.
   - Medical data: count of symptoms or test results.

**When *Not* to Use It**

- If your features are **binary (0/1)** instead use **Bernoulli Naive Bayes** instead.  
- If your features are **continuous values** instead use **Gaussian Naive Bayes**.  
- If your features are highly dependent on each other (e.g., strong correlations), Naive Bayes models in general may not perform well.

In [ ]:
np.random.seed(42)
n_samples = 200

# Feature 1: mostly high counts for class 1
X1_class0 = np.random.randint(0, 3, size=(50, 1))
X1_class1 = np.random.randint(5, 10, size=(50, 1))
# Feature 2: mostly high counts for class 0
X2_class0 = np.random.randint(5, 10, size=(50, 1))
X2_class1 = np.random.randint(0, 3, size=(50, 1))

X = np.vstack([np.hstack([X1_class0, X2_class0]), np.hstack([X1_class1, X2_class1])])
y = np.array([0]*50 + [1]*50)

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(X[y == 0, 0], X[y == 0, 1], color='steelblue', label='Class 0', s=70)
plt.scatter(X[y == 1, 0], X[y == 1, 1], color='darkorange', label='Class 1', s=70)
plt.xlabel("Feature 1 (count-like)")
plt.ylabel("Feature 2 (count-like)")
plt.title("Synthetic Count-Based Data")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

In [ ]:
mnb = MultinomialNB()
mnb.fit(X, y)
plot_decision_boundary(mnb, X, y, "Multinomial Naive Bayes Decision Boundary")

In [ ]:
y_pred = mnb.predict(X)
print("Accuracy:", accuracy_score(y, y_pred))

## Application - Spam Collection

Here we will explore a more realistic example of when we might use Naive Bayes' Classification. 

*From https://www.kaggle.com/code/jeffysonar/spam-filter-using-naive-bayes-classifier*

The SMS Spam Collection is a set of SMS tagged messages that have been collected for SMS Spam research. It contains one set of SMS messages in English of 5,574 messages, tagged according to being ham (legitimate) or spam.

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("uciml/sms-spam-collection-dataset")

print("Path to dataset files:", path)

file = path + '/' + os.listdir(path)[0]
df = pd.read_csv(file, encoding='latin-1')[['v1', 'v2']]
df.columns = ['label', 'message']
df.head()

In [ ]:
df.groupby('label').describe()

In [ ]:
df['label'].value_counts()

## Text Based Data

When using data that is text based there is quite a bit of cleaning that you need to do. We need to 

**Clean and normalize text**

- Remove punctuations
- Remove all stopwords
- Apply stemming (converting to normal form of word).For example, 'driving car' and 'drives car' becomes drive car

**Convert the text into vectors**

This will let us pass numbers into our model so it understand the text.

**Train and test our classifier**


## Clean and Normalize

We will use the NLTK package! You already installed it on Day 1 with `pip install -r requirements.txt`, so there is nothing to install today. NLTK stands for Natural Language Toolkit. It is used for:

- Tokenizing text into words or sentences

- Removing stopwords

- Stemming or lemmatization

- Part-of-speech tagging

- Parsing and text classification

- Working with corpora and linguistic datasets

It is widely used in teaching and prototyping Natural Language Processing NLP projects because it provides ready-to-use tools and datasets.

In [ ]:
import string
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
from nltk import PorterStemmer as Stemmer
def process(text):
    '''
    Preprocess text by first making sure it is lower case.
    Then remove punctuation and words that are too common (stopwords)

    Stopwords are common words in a language that are usually filtered 
    out before processing text because they carry little semantic meaning for many tasks
    '''
    # lowercase it
    text = text.lower()
    # remove punctuation
    text = ''.join([t for t in text if t not in string.punctuation])
    # remove stopwords
    text = [t for t in text.split() if t not in stopwords.words('english')]
    # stemming
    st = Stemmer()
    text = [st.stem(t) for t in text]
    # return token list
    return text

In [ ]:
process('It\'s holiday and we are playing cricket. Jeff is playing very well!!!')

We can apply this processing to the first twenty lines of our data to see what is created.

In [ ]:
df['message'][:20].apply(process)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(df['message'], 
                                                    df['label'], 
                                                    test_size=0.20, 
                                                    random_state = 42)

## Convert to vectors

Machine learning models need **numbers**, so we have to convert words into vectors. There are lots of ways to convert words to numbers:

1. One-Hot Encoding
- Each word → a vector of 0s and a single 1 for that word.
- Simple but **sparse** and ignores similarity.

2. Bag-of-Words (Count Vectors)
- Count how many times each word appears in a document.
- Keeps frequency info but **ignores word order**.

3. TF-IDF
- Like Bag-of-Words but **weights words by importance**.
- Down-weights common words like "the" or "and".

4. Word Embeddings
- Each word → dense vector capturing **meaning**.
- Similar words have **similar vectors**.
- Examples: Word2Vec, GloVe, FastText.

5. Contextual Embeddings
- Word vectors **depend on the sentence**.
- Examples: BERT, GPT.
- Captures different meanings of the same word depending on context.

**We will use Bag-of-words!**

We will use the TfidfVectorizer from sklearn to create these features. TF (Term Frequency): How often a word appears in a document. IDF (Inverse Document Frequency): Down-weights words that appear in many documents (common words like “the”, “and”).

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
# Create the vectorizer
# Tell it to use the process function from above
tfidfv = TfidfVectorizer(analyzer=process)
# Fit and convert our messages
data_train = tfidfv.fit_transform(X_train)

In [ ]:
# Let's look at what is in here
words = X_train.iloc[0]
words

In [ ]:
print(tfidfv.transform([words]))

In [ ]:
# Create the model
mnb = MultinomialNB()
mnb.fit(data_train,y_train)

In [ ]:
data_test = tfidfv.transform(X_test)
preds=mnb.predict(data_test)

In [ ]:
count = 0
for i in range(len(y_test)):
    if y_test.iloc[i] != preds[i]:
        count += 1
print('Total number of test cases', len(y_test))
print('Number of wrong of predictions', count)

In [ ]:
print(classification_report(y_test, preds))

Look at the spam row. The precision for spam is 1.00: every message the model called spam really was spam. The recall for spam is lower, so the mistakes we made were all spam messages that slipped through as ham. That is the mistake we would rather make! The cost of a spam message landing in your inbox is small next to the cost of a real message from a friend going into the spam folder. This is a case where we want high precision on the spam category.

In [ ]:
def detect_spam(words):
    d = tfidfv.transform([words])
    pred=mnb.predict(d)
    return str(pred[0])

In [ ]:
detect_spam('Your cash-balance is currently 500 pounds - to maximize your cash-in now, send COLLECT to 83600.')

In [ ]:
detect_spam('your bank acount immediately send now to get your money')

---------------------------------------

## Homework 14

The full assignment is in `HW_day14.ipynb` in your sandbox. You get to choose: build a spam detector like the one in these notes, or use a Naive Bayes classifier to predict survival on the Titanic.

Work the problems in your sandbox and push it, your sandbox is your backup. Your write-up notebook goes in `Week08` of your team's repo, along with Homework 13. Both are due **Sunday 11/8 at 11:59pm**.